# Customer Churn Prediction - Preprocessing and Modeling

## Objective
The goal of this notebook is to prepare the cleaned dataset for machine learning and build baseline classification models to predict customer churn.

In [62]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

## Load the Dataset

We load the Telco Customer Churn dataset again in this notebook.

The same initial cleaning step for `TotalCharges` will be applied before preprocessing and modeling.

In [63]:
df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [64]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Initial Cleaning

Before modeling, we apply the same cleaning step identified during data understanding.

The `TotalCharges` column is converted from text to numeric, and missing values caused by blank entries are replaced with 0.

In [65]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [66]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

## Define Features and Target

Before training a model, we separate the dataset into:

- `X`: the input features used to make predictions
- `y`: the target variable we want to predict

The target variable is `Churn`.

The `customerID` column is removed because it is only an identifier and does not provide useful predictive information.

In [67]:
X = df.drop(["customerID", "Churn"], axis=1)
y = df["Churn"]

In [68]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 19)
y shape: (7043,)


## Encode the Target Variable

The target variable `Churn` contains text values: `Yes` and `No`.

For machine learning, we convert it into numerical values:

- `Yes` becomes 1
- `No` becomes 0

In [69]:
y = y.map({"No": 0, "Yes": 1})

In [70]:
y.value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

## Train-Test Split

The dataset is split into training and testing sets.

The training set is used to train the model, while the testing set is kept aside to evaluate model performance on unseen data.

We use `stratify=y` to keep the same churn/non-churn proportion in both sets.

In [71]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [72]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (5634, 19)
X_test shape: (1409, 19)
y_train shape: (5634,)
y_test shape: (1409,)


## Identify Numerical and Categorical Features

Before preprocessing, we separate the input features into numerical and categorical columns.

Numerical features will be scaled.

Categorical features will be encoded using one-hot encoding.

In [73]:
numerical_features = X_train.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X_train.select_dtypes(include=["object", "string"]).columns

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='str')

Categorical features:
Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='str')


## Build the Preprocessing Pipeline

Numerical features are scaled using `StandardScaler`.

Categorical features are converted into numerical format using `OneHotEncoder`.

The `ColumnTransformer` applies the correct preprocessing to each type of column.

In [74]:
numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## Baseline Model: Dummy Classifier

Before training real machine learning models, we create a baseline model using `DummyClassifier`.

This model provides a simple reference point. Any useful model should perform better than this baseline.

In [75]:
dummy_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent"))
])

In [76]:
dummy_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

## Evaluate the Dummy Classifier

The dummy model predicts the most frequent class every time.

This gives us a baseline performance. A real machine learning model should perform better than this.

In [77]:
y_pred_dummy = dummy_model.predict(X_test)

In [78]:
print(classification_report(y_test, y_pred_dummy))

              precision    recall  f1-score   support

           0       0.73      1.00      0.85      1035
           1       0.00      0.00      0.00       374

    accuracy                           0.73      1409
   macro avg       0.37      0.50      0.42      1409
weighted avg       0.54      0.73      0.62      1409



c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

In [79]:
confusion_matrix(y_test, y_pred_dummy)

array([[1035,    0],
       [ 374,    0]])

## Interpretation: Dummy Classifier

The dummy classifier predicts the majority class every time.

Since most customers did not churn, the dummy model predicts `No churn` for almost all or all customers.

This model may have acceptable accuracy because the dataset is imbalanced, but it fails to identify churned customers.

The recall for the churn class is expected to be 0, meaning the model does not correctly detect customers who actually churned.

This confirms that accuracy alone is not enough for this project. We need to evaluate future models using precision, recall, F1-score, and ROC-AUC.

## Model 1: Logistic Regression

Logistic Regression is used as the first real classification model.

It is a good baseline model because it is simple, fast, and easy to interpret.

In [80]:
log_reg_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [81]:
log_reg_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

## Evaluate Logistic Regression

The Logistic Regression model is evaluated on the test set.

We compare its results with the Dummy Classifier to check whether it performs better than the baseline.

In [82]:
y_pred_log_reg = log_reg_model.predict(X_test)

In [83]:
print(classification_report(y_test, y_pred_log_reg))

              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



In [84]:
confusion_matrix(y_test, y_pred_log_reg)

array([[926, 109],
       [165, 209]])

In [85]:
y_proba_log_reg = log_reg_model.predict_proba(X_test)[:, 1]

In [86]:
roc_auc_score(y_test, y_proba_log_reg)

0.8421349040274871

## Logistic Regression ROC-AUC

ROC-AUC measures how well the model separates churned customers from non-churned customers.

A value closer to 1 means better separation.

This metric is useful because the dataset is imbalanced and accuracy alone may be misleading.

## Interpretation: Logistic Regression

Logistic Regression performs better than the Dummy Classifier because it can identify some customers who churned.

The model should be evaluated using multiple metrics:

- Accuracy shows the overall percentage of correct predictions.
- Precision shows, among customers predicted as churn, how many actually churned.
- Recall shows, among customers who actually churned, how many the model correctly detected.
- F1-score balances precision and recall.
- ROC-AUC shows how well the model separates churned customers from non-churned customers.

For this project, recall for the churn class is especially important because missing customers who are likely to churn can be costly for the company.

In [87]:
class_weight="balanced"

## Model 2: Logistic Regression with Class Weight

Because the dataset is imbalanced, we train another Logistic Regression model using `class_weight="balanced"`.

This gives more importance to the minority class, which is the churn class.
The goal is to improve recall for customers who churned.

In [88]:
log_reg_balanced_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

In [89]:
log_reg_balanced_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

## Evaluate Logistic Regression with Class Weight

This model is evaluated to check whether using `class_weight="balanced"` improves the detection of churned customers.

The main metric to watch is recall for class 1, which represents customers who churned.

In [90]:
y_pred_log_reg_balanced = log_reg_balanced_model.predict(X_test)

In [91]:
print(classification_report(y_test, y_pred_log_reg_balanced))

              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409



In [92]:
confusion_matrix(y_test, y_pred_log_reg_balanced)

array([[747, 288],
       [ 81, 293]])

In [93]:
y_proba_log_reg_balanced = log_reg_balanced_model.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, y_proba_log_reg_balanced)

0.8416388953473353

## Compare Logistic Regression Models

In this section, we compare the standard Logistic Regression model with the class-weighted Logistic Regression model.

The goal is to see whether class weighting improves churn detection, especially recall for class 1.

In [94]:
print("Standard Logistic Regression:")
print(classification_report(y_test, y_pred_log_reg))

print("\nBalanced Logistic Regression:")
print(classification_report(y_test, y_pred_log_reg_balanced))

Standard Logistic Regression:
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409


Balanced Logistic Regression:
              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409



## Interpretation: Standard vs Balanced Logistic Regression

The balanced Logistic Regression model usually improves recall for the churn class.

However, this may reduce precision, meaning the model may predict more customers as churned even if some of them did not actually churn.

This is an important business tradeoff:

- Higher recall means fewer churned customers are missed.
- Lower precision means more false alarms.

For churn prediction, higher recall can be valuable because the company wants to identify as many at-risk customers as possible.

## Model Comparison Table

In this section, we create a comparison table to summarize the performance of the models tested so far.

In [95]:
model_results = pd.DataFrame({
    "Model": [
        "Dummy Classifier",
        "Logistic Regression",
        "Logistic Regression Balanced"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_dummy),
        accuracy_score(y_test, y_pred_log_reg),
        accuracy_score(y_test, y_pred_log_reg_balanced)
    ],
    "Precision": [
        precision_score(y_test, y_pred_dummy, zero_division=0),
        precision_score(y_test, y_pred_log_reg),
        precision_score(y_test, y_pred_log_reg_balanced)
    ],
    "Recall": [
        recall_score(y_test, y_pred_dummy, zero_division=0),
        recall_score(y_test, y_pred_log_reg),
        recall_score(y_test, y_pred_log_reg_balanced)
    ],
    "F1-score": [
        f1_score(y_test, y_pred_dummy, zero_division=0),
        f1_score(y_test, y_pred_log_reg),
        f1_score(y_test, y_pred_log_reg_balanced)
    ],
    "ROC-AUC": [
        np.nan,
        roc_auc_score(y_test, y_proba_log_reg),
        roc_auc_score(y_test, y_proba_log_reg_balanced)
    ]
})

model_results

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Dummy Classifier,0.734564,0.000000,0.000000,0.000000,NaN
1,Logistic Regression,0.805536,0.657233,0.558824,0.604046,0.842135
2,Logistic Regression Balanced,0.738112,0.504303,0.783422,0.613613,0.841639


## Interpretation: Model Comparison

The Dummy Classifier is only a baseline and does not identify churned customers well.

The standard Logistic Regression model performs better because it learns patterns from the data.

The balanced Logistic Regression model usually improves recall for the churn class, meaning it detects more customers who actually churned.

However, improving recall may reduce precision, meaning the model may create more false positives.

For this project, recall for class 1 is very important because class 1 represents customers who churned.

In [96]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from xgboost import XGBClassifier

## Model 3: Decision Tree

A Decision Tree is a classification model that splits the data based on feature values.

It is easy to understand, but it can overfit if the tree becomes too deep.

In [97]:
decision_tree_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

In [98]:
decision_tree_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

## Evaluate Decision Tree

The Decision Tree model is evaluated on the test set.

We compare its performance with Logistic Regression models to see whether it improves churn prediction.

In [99]:
y_pred_tree = decision_tree_model.predict(X_test)

In [100]:
print(classification_report(y_test, y_pred_tree))

              precision    recall  f1-score   support

           0       0.81      0.80      0.81      1035
           1       0.48      0.49      0.48       374

    accuracy                           0.72      1409
   macro avg       0.64      0.65      0.65      1409
weighted avg       0.72      0.72      0.72      1409



In [101]:
confusion_matrix(y_test, y_pred_tree)

array([[832, 203],
       [190, 184]])

In [102]:
y_proba_tree = decision_tree_model.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, y_proba_tree)

0.6476762510010592

## Interpretation: Decision Tree

The Decision Tree model can capture more complex relationships than Logistic Regression.

However, Decision Trees can easily overfit, especially when no maximum depth is specified.

If the Decision Tree performs worse than Logistic Regression on the test set, this may indicate overfitting or poor generalization.

We will later compare all models using the same metrics to choose the best model.

## Model 4: Random Forest

Random Forest is an ensemble model that combines many decision trees.

It usually performs better than a single Decision Tree because it reduces overfitting and improves generalization.

In [103]:
random_forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

In [104]:
random_forest_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

## Evaluate Random Forest

The Random Forest model is evaluated on the test set.

We compare its performance with Logistic Regression and Decision Tree models.

In [105]:
y_pred_rf = random_forest_model.predict(X_test)

In [106]:
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.49      0.55       374

    accuracy                           0.79      1409
   macro avg       0.73      0.69      0.70      1409
weighted avg       0.77      0.79      0.78      1409



In [107]:
confusion_matrix(y_test, y_pred_rf)

array([[926, 109],
       [192, 182]])

In [108]:
y_proba_rf = random_forest_model.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, y_proba_rf)

0.8185099072567104

## Interpretation: Random Forest

The Random Forest model combines many decision trees, which usually makes it more stable than a single Decision Tree.

Compared to the Decision Tree, Random Forest may reduce overfitting and improve generalization.

However, the model should still be compared using recall, F1-score, and ROC-AUC, not accuracy alone.

For churn prediction, we are especially interested in how well the model detects class 1, which represents customers who churned.

## Interpretation: Random Forest

The Random Forest model combines many decision trees, which usually makes it more stable than a single Decision Tree.

Compared to the Decision Tree, Random Forest may reduce overfitting and improve generalization.

However, the model should still be compared using recall, F1-score, and ROC-AUC, not accuracy alone.

For churn prediction, we are especially interested in how well the model detects class 1, which represents customers who churned.

In [109]:
gradient_boosting_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(random_state=42))
])

In [110]:
gradient_boosting_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

## Evaluate Gradient Boosting

The Gradient Boosting model is evaluated on the test set.

We compare its performance with the previous models to see whether it improves churn prediction.

In [111]:
y_pred_gb = gradient_boosting_model.predict(X_test)

In [112]:
print(classification_report(y_test, y_pred_gb))

              precision    recall  f1-score   support

           0       0.84      0.91      0.87      1035
           1       0.67      0.52      0.58       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.73      1409
weighted avg       0.79      0.80      0.79      1409



In [113]:
confusion_matrix(y_test, y_pred_gb)

array([[938,  97],
       [181, 193]])

In [114]:
y_proba_gb = gradient_boosting_model.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, y_proba_gb)

0.8432754656539823

## Interpretation: Gradient Boosting

Gradient Boosting is a stronger model that builds trees sequentially, with each tree trying to correct the errors of the previous ones.

It may perform better than Logistic Regression, Decision Tree, or Random Forest, especially on structured tabular data.

However, we should compare it using multiple metrics, especially recall, F1-score, and ROC-AUC.

For this churn project, the best model is not necessarily the one with the highest accuracy. The best model should detect churned customers well while keeping false positives reasonably controlled.

## Model 6: XGBoost

XGBoost is a powerful gradient boosting algorithm that often performs very well on structured/tabular datasets.

It is commonly used in machine learning competitions and real-world classification problems.

In [115]:
xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ))
])

In [116]:
xgb_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

## Evaluate XGBoost

The XGBoost model is evaluated on the test set.

We compare its performance with Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting.

In [117]:
y_pred_xgb = xgb_model.predict(X_test)

In [118]:
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.83      0.87      0.85      1035
           1       0.58      0.51      0.54       374

    accuracy                           0.77      1409
   macro avg       0.71      0.69      0.70      1409
weighted avg       0.76      0.77      0.77      1409



In [119]:
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, y_proba_xgb)

0.8151941408974658

## Interpretation: XGBoost

XGBoost is a powerful boosting model that can capture complex patterns in the data.

Its performance should be compared with the previous models using accuracy, precision, recall, F1-score, and ROC-AUC.

For this churn project, we should pay special attention to recall for class 1, because class 1 represents customers who churned.

Even if XGBoost has strong overall performance, it is important to check whether it actually improves churn detection compared to simpler models.

## Updated Model Comparison Table

In this section, we update the model comparison table by adding the advanced models.

This allows us to compare all models using the same evaluation metrics.

In [120]:
model_results = pd.DataFrame({
    "Model": [
        "Dummy Classifier",
        "Logistic Regression",
        "Logistic Regression Balanced",
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting",
        "XGBoost"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_dummy),
        accuracy_score(y_test, y_pred_log_reg),
        accuracy_score(y_test, y_pred_log_reg_balanced),
        accuracy_score(y_test, y_pred_tree),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_gb),
        accuracy_score(y_test, y_pred_xgb)
    ],
    "Precision": [
        precision_score(y_test, y_pred_dummy, zero_division=0),
        precision_score(y_test, y_pred_log_reg),
        precision_score(y_test, y_pred_log_reg_balanced),
        precision_score(y_test, y_pred_tree),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_gb),
        precision_score(y_test, y_pred_xgb)
    ],
    "Recall": [
        recall_score(y_test, y_pred_dummy, zero_division=0),
        recall_score(y_test, y_pred_log_reg),
        recall_score(y_test, y_pred_log_reg_balanced),
        recall_score(y_test, y_pred_tree),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_gb),
        recall_score(y_test, y_pred_xgb)
    ],
    "F1-score": [
        f1_score(y_test, y_pred_dummy, zero_division=0),
        f1_score(y_test, y_pred_log_reg),
        f1_score(y_test, y_pred_log_reg_balanced),
        f1_score(y_test, y_pred_tree),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_gb),
        f1_score(y_test, y_pred_xgb)
    ],
    "ROC-AUC": [
        np.nan,
        roc_auc_score(y_test, y_proba_log_reg),
        roc_auc_score(y_test, y_proba_log_reg_balanced),
        roc_auc_score(y_test, y_proba_tree),
        roc_auc_score(y_test, y_proba_rf),
        roc_auc_score(y_test, y_proba_gb),
        roc_auc_score(y_test, y_proba_xgb)
    ]
})

model_results.sort_values(by="Recall", ascending=False)

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
2,Logistic Regression Balanced,0.738112,0.504303,0.783422,0.613613,0.841639
1,Logistic Regression,0.805536,0.657233,0.558824,0.604046,0.842135
5,Gradient Boosting,0.802697,0.665517,0.516043,0.581325,0.843275
6,XGBoost,0.772889,0.582317,0.510695,0.544160,0.815194
3,Decision Tree,0.721079,0.475452,0.491979,0.483574,0.647676
4,Random Forest,0.786373,0.625430,0.486631,0.547368,0.818510
0,Dummy Classifier,0.734564,0.000000,0.000000,0.000000,NaN


## Interpretation: Full Model Comparison

The models are compared using accuracy, precision, recall, F1-score, and ROC-AUC.

Because this is a churn prediction problem, recall for class 1 is especially important. A higher recall means the model detects more customers who actually churned.

However, recall should not be considered alone. A model with very high recall but very low precision may create too many false positives.

The best model should provide a good balance between:
- detecting churned customers
- avoiding too many false alarms
- maintaining strong overall performance

At this stage, the best candidate model should be selected based on the business goal, not only on accuracy.

## Cross-Validation Setup

Cross-validation gives a more reliable estimate of model performance than a single train-test split.

We use `StratifiedKFold` because the target variable is imbalanced. This keeps the churn and non-churn proportions similar in each fold.

In [121]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

## Cross-Validation Model Comparison

We create a dictionary of model pipelines.

Each model uses the same preprocessing pipeline, so the comparison is fair.

In [122]:
cv_models = {
    "Logistic Regression": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]),

    "Logistic Regression Balanced": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
    ]),

    "Random Forest": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42))
    ]),

    "Gradient Boosting": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", GradientBoostingClassifier(random_state=42))
    ]),

    "XGBoost": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(
            random_state=42,
            eval_metric="logloss"
        ))
    ])
}

## Run Cross-Validation

We evaluate each model using 5-fold cross-validation on the training data.

This gives a more reliable estimate of model performance than using only one train-test split.

We evaluate the models using recall, precision, F1-score, and ROC-AUC.

In [123]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results = []

for model_name, model in cv_models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )
    
    cv_results.append({
        "Model": model_name,
        "CV Accuracy": scores["test_accuracy"].mean(),
        "CV Precision": scores["test_precision"].mean(),
        "CV Recall": scores["test_recall"].mean(),
        "CV F1-score": scores["test_f1"].mean(),
        "CV ROC-AUC": scores["test_roc_auc"].mean()
    })

cv_results_df = pd.DataFrame(cv_results)
cv_results_df.sort_values(by="CV Recall", ascending=False)

,Model,CV Accuracy,CV Precision,CV Recall,CV F1-score,CV ROC-AUC
1,Logistic Regression Balanced,0.748493,0.516933,0.801338,0.628257,0.845953
0,Logistic Regression,0.802096,0.652927,0.543144,0.592329,0.846201
3,Gradient Boosting,0.803163,0.661703,0.529097,0.587769,0.848129
4,XGBoost,0.784880,0.613883,0.512375,0.558269,0.822698
2,Random Forest,0.785235,0.626002,0.474247,0.539393,0.817815


## Interpretation: Cross-Validation Results

Cross-validation provides a more reliable comparison between models because each model is evaluated across multiple folds of the training data.

For this churn prediction project, recall is especially important because it measures how many actual churned customers the model correctly identifies.

However, recall should not be considered alone. A model with very high recall but low precision may create too many false positives.

The best model should provide a good balance between recall, F1-score, and ROC-AUC.

The final model should be selected based on both cross-validation performance and business priorities.